<a href="https://colab.research.google.com/github/zhakhinaaaa/titanic-survival/blob/main/03_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio

In [1]:
import urllib.request, json, joblib
import numpy as np, pandas as pd
from tensorflow import keras
import gradio as gr

USER = "zhakhinaaaa"
MODELS = f"https://raw.githubusercontent.com/{USER}/titanic-survival/main/models/"

for f in ["titanic_model.keras", "scaler.pkl", "columns.json"]:
    urllib.request.urlretrieve(MODELS + f, f)

model = keras.models.load_model("titanic_model.keras")
scaler = joblib.load("scaler.pkl")
columns = json.load(open("columns.json"))
print("Модель загружена, признаков:", len(columns))   # должно быть 17

Модель загружена, признаков: 17


In [2]:
PORTS = {"Саутгемптон (S)": "S", "Шербур (C)": "C", "Квинстаун (Q)": "Q"}
MEDIAN_FARE = {1: 60.29, 2: 14.25, 3: 8.05}   # медианы из train

def build_features(pclass, sex, age, sibsp, parch, fare, port, has_cabin, married):
    pclass, sibsp, parch = int(pclass), int(sibsp), int(parch)
    male = sex == "Мужской"
    if male:
        title = "Master" if age < 13 else "Mr"
    else:
        title = "Mrs" if married else "Miss"
    if not fare or fare <= 0:              # цена не указана → медиана по классу
        fare = MEDIAN_FARE[pclass]

    row = dict.fromkeys(columns, 0)
    row.update({"Pclass": pclass, "Sex": int(male), "Age": age,
                "SibSp": sibsp, "Parch": parch, "Fare": fare,
                "HasCabin": int(has_cabin),
                "FamilySize": sibsp + parch + 1,
                "IsAlone": int(sibsp + parch == 0)})
    row["Embarked_" + PORTS[port]] = 1
    row["Title_" + title] = 1
    return pd.DataFrame([row])[columns]

def predict(pclass, sex, age, sibsp, parch, fare, port, has_cabin, married):
    X = build_features(pclass, sex, age, sibsp, parch, fare, port, has_cabin, married)
    p = float(model.predict(scaler.transform(X), verbose=0)[0][0])
    verdict = "Скорее всего, вы бы **выжили**" if p >= 0.5 else "Скорее всего, вы бы **не выжили**"
    return {"Выжил": p, "Погиб": 1 - p}, f"### {verdict}\nВероятность спасения: **{p:.0%}**"

# быстрая проверка
print(predict("1", "Женский", 30, 1, 0, 0, "Шербур (C)", True, True))
print(predict("3", "Мужской", 25, 0, 0, 0, "Саутгемптон (S)", False, False))

({'Выжил': 0.9438730478286743, 'Погиб': 0.056126952171325684}, '### Скорее всего, вы бы **выжили**\nВероятность спасения: **94%**')
({'Выжил': 0.098103828728199, 'Погиб': 0.901896171271801}, '### Скорее всего, вы бы **не выжили**\nВероятность спасения: **10%**')


In [3]:
demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Radio(["1", "2", "3"], value="3", label="Класс билета"),
        gr.Radio(["Мужской", "Женский"], value="Мужской", label="Пол"),
        gr.Slider(0, 80, value=25, step=1, label="Возраст"),
        gr.Slider(0, 8, value=0, step=1, label="Братьев/сестёр и супругов на борту (SibSp)"),
        gr.Slider(0, 6, value=0, step=1, label="Родителей и детей на борту (Parch)"),
        gr.Number(value=0, label="Цена билета, £ (0 — подставить типичную для класса)"),
        gr.Dropdown(list(PORTS), value="Саутгемптон (S)", label="Порт посадки"),
        gr.Checkbox(label="Есть каюта"),
        gr.Checkbox(label="Замужем (только для женщин)"),
    ],
    outputs=[gr.Label(num_top_classes=2, label="Прогноз нейросети"), gr.Markdown()],
    title="Спасся бы ты на Титанике?",
    description="Введите данные пассажира — нейросеть, обученная на данных Kaggle, оценит шанс выжить.",
    examples=[
        ["1", "Женский", 30, 1, 0, 80, "Шербур (C)", True, True],
        ["3", "Мужской", 25, 0, 0, 8, "Саутгемптон (S)", False, False],
        ["2", "Мужской", 6, 1, 2, 0, "Саутгемптон (S)", False, False],
    ],
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://40267431c1b352800f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
